<a href="https://colab.research.google.com/github/John-Fernand/TesteGit/blob/main/AtividadeWebScraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install -q google-colab-selenium

In [8]:
import re
import google_colab_selenium as gs
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

In [15]:
cidade = input("Digite a cidade e o estado: ").strip()
cidade_busca = "".join(c for c in unicodedata.normalize("NFD", cidade.casefold()) if unicodedata.category(c) != "Mn")
navegador = gs.Chrome()
espera = WebDriverWait(navegador, 30)
try:
    navegador.get("https://www.climatempo.com.br/previsao-do-tempo")
    links = espera.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "a[href*='/previsao-do-tempo/agora/cidade/']")))
    url_cidade = None
    for link in links:
        nome = "".join(c for c in unicodedata.normalize("NFD", link.text.casefold()) if unicodedata.category(c) != "Mn")
        if nome.strip() == cidade_busca:
            url_cidade = link.get_attribute("href")
            break
    if url_cidade:
        url_cidade = url_cidade.replace("/agora/cidade/", "/cidade/")
        navegador.get(url_cidade)
        espera.until(EC.text_to_be_present_in_element((By.TAG_NAME, "body"), "Qual a previsão do tempo para hoje"))
        texto = navegador.find_element(By.TAG_NAME, "body").text
        resultado = re.search(r"temperaturas?\s+entre\s+(-?\d{1,2})°\s+e\s+(-?\d{1,2})°", texto, re.IGNORECASE)
        if resultado:
            minima = resultado.group(1)
            maxima = resultado.group(2)
            print("\nPrevisão do tempo")
            print("Cidade:", cidade)
            print("Temperatura mínima:", minima, "°C")
            print("Temperatura máxima:", maxima, "°C")
        else:
            print("Temperaturas não encontradas.")
    else:
        print("Cidade não encontrada na lista do Climatempo.")
except TimeoutException:
    print("Tempo limite excedido ao carregar a página.")
finally:
    navegador.quit()


Digite a cidade e o estado: Santa Maria, RS


<IPython.core.display.Javascript object>


Previsão do tempo
Cidade: Santa Maria, RS
Temperatura mínima: 19 °C
Temperatura máxima: 28 °C
